# YOLOv7 — PhenoBench plant detection

Trains **YOLOv7** on PhenoBench multiclass (`crop` / `weed`) plant detection,
reproducing the detection baseline from **Weyler et al. 2024, Table 5**:

| Approach | mAP | mAP₅₀ | mAP₇₅ | AP crop | AP weed |
|---|---|---|---|---|---|
| Faster R-CNN | 40.43 | 65.07 | 40.19 | 63.23 | 17.62 |
| Mask R-CNN | 38.68 | 63.72 | 38.07 | 60.32 | 17.05 |
| **YOLOv7** | **60.48** | **82.47** | **62.30** | **83.06** | **37.91** |

One notebook, two **parameterized** variants (see the config cell):

- **`full`** — reproduces the baseline: full YOLOv7 (37.2 M params) on the full
  1024² images. `img-size` defaults to YOLOv7's standard **640** (the paper's
  exact detection resolution is only in the supplement; bump to 1024 to push AP).
- **`tiny`** — the NPU-bound variant: **YOLOv7-tiny** on **2×2 = 512² tiles**.
  Tiling keeps the weeds at native pixel density while landing on a 512 input
  that runs well on the i.MX targets. This will *not* match the Table 5 number;
  it is the deployment-oriented follow-up.

Data is materialized straight from the **raw PhenoBench images** via the
project's `data.yolo` exporter (reusing the `phenobench` dataloader and the
canonical `DatasetDefinition`) into the on-disk layout YOLOv7's own dataloader
expects — no pre-baked dataset bundle.

> **Caveat.** The Table 5 numbers are on PhenoBench's *hidden test set*. The
> local `test` split has no labels, so this notebook self-evaluates on **`val`**
> — treat the printed metrics as a reproduction *proxy*, not the leaderboard
> number. The paper also marks predictions on partially-visible instances
> (<50 % in-frame) as *do-not-care* at eval; standard YOLO eval cannot express
> that, a second reason val AP here reads slightly lower.

## 1 · Environment

Target runtime: **Kaggle**, GPU enabled. **Select the Python 3.10 image**
("Pin to original environment"), *not* the latest. The choice is binary on
Kaggle — pinned 3.10 or latest 3.12 — and for YOLOv7 the older one is correct:
the 3.10 image ships **numpy <1.24** and **torch 1.x/2.0**, the stack YOLOv7
(WongKinYiu, 2022) was written against, so it runs as-is. The latest 3.12 image
brings **numpy 2.x**, which *removed* `np.int` / `np.float`; YOLOv7's source
still uses them, so it raises `AttributeError`s until you patch its files. This
is a pure-PyTorch run with **no TensorFlow**, so the project's `>=3.10,<3.11`
pin (which exists only for TF 2.12) is irrelevant here — 3.10 is chosen for
YOLOv7's sake, not the repo's. Torch / OpenCV come from the image; we add only
YOLOv7's extras, the `phenobench` dataloader, and this project (`--no-deps`, so
no TensorFlow stack).

In [ ]:
!python -V
!nvidia-smi -L

In [ ]:
# YOLOv7 (official implementation).
!git clone --depth 1 https://github.com/WongKinYiu/yolov7.git

# YOLOv7 extras not always present on Kaggle, plus this project (only used here
# to regenerate data.yaml against the mounted dataset -> no TensorFlow, and no
# phenobench: materialization lives in the export notebook).
!pip install --no-cache-dir thop phenobench
!pip install --no-cache-dir --no-deps git+https://github.com/frdiener/agri-vision-edge.git

## 2 · Configuration

Flip `VARIANT` to switch between the baseline reproduction and the tiled tiny
variant. Each maps to a sub-directory of the mounted **`phenobench-yolo`**
dataset built by `phenobench_yolo_dataset_export.ipynb`.

In [ ]:
from pathlib import Path

# "full" -> reproduce the Table 5 baseline; "tiny" -> NPU-bound tiled variant.
VARIANT = "tiny"

# Mounted YOLO dataset produced by phenobench_yolo_dataset_export.ipynb.
YOLO_DS_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/phenobench-yolo"
)

WORK = Path("/kaggle/working")
RUNS_DIR = WORK / "runs"                    # YOLOv7 training output
RUN_NAME = f"yolov7_{VARIANT}_phenobench"

_PRESETS = {
    "full": dict(
        dataset_subdir="full",             # full 1024 images
        img=640,                           # YOLOv7 default; set 1024 to push AP
        cfg="cfg/training/yolov7.yaml",
        weights="yolov7.pt",
        hyp="data/hyp.scratch.custom.yaml",
        batch=16,
        epochs=100,
    ),
    "tiny": dict(
        dataset_subdir="tiled512",         # 2×2 = 512² tiles
        img=512,
        cfg="cfg/training/yolov7-tiny.yaml",
        weights="yolov7-tiny.pt",
        hyp="data/hyp.scratch.tiny.yaml",
        batch=32,
        epochs=100,
    ),
}

CFG_PRESET = _PRESETS[VARIANT]
DATASET_DIR = YOLO_DS_ROOT / CFG_PRESET["dataset_subdir"]
IMG = CFG_PRESET["img"]
CFG = CFG_PRESET["cfg"]
WEIGHTS = CFG_PRESET["weights"]
HYP = CFG_PRESET["hyp"]
BATCH = CFG_PRESET["batch"]
EPOCHS = CFG_PRESET["epochs"]

assert DATASET_DIR.exists(), f"missing mount: {DATASET_DIR} -- attach the phenobench-yolo dataset"
print(f"VARIANT={VARIANT}  img={IMG}  dataset={DATASET_DIR}  cfg={CFG}")

## 3 · Point YOLOv7 at the dataset (writable view)

The images + labels are already materialized in the mount, but YOLOv7 writes a
`*.cache` file **next to the labels** — and `/kaggle/input` is read-only, so
training would crash with `… train.cache cannot be opened`.

So we build a thin **writable view** under the working dir: `images/` and
`labels/` are real directories whose `train` / `val` sub-dirs **symlink** back
to the mount (no copy — instant). The cache then lands in a writable directory.
`data.yaml` points at this view with **absolute** `train` / `val` paths (YOLOv7
ignores the ultralytics `path:` key and resolves these against its own cwd, so
they must be absolute).

In [ ]:
import os
from agri_vision_edge.data import (
    PHENOBENCH_MULTICLASS,
    write_data_yaml,
    yolo_class_names,
)

DATASET_DEF = PHENOBENCH_MULTICLASS
class_names = yolo_class_names(DATASET_DEF)  # ["crop", "weed"]

# Writable view: real images/ + labels/ dirs, split sub-dirs symlinked to mount.
VIEW_DIR = WORK / "ds" / CFG_PRESET["dataset_subdir"]
for sub in ("images", "labels"):
    (VIEW_DIR / sub).mkdir(parents=True, exist_ok=True)
    for split in ("train", "val"):
        link = VIEW_DIR / sub / split
        if link.is_symlink() or link.exists():
            continue
        os.symlink(DATASET_DIR / sub / split, link)

DATA_YAML = write_data_yaml(
    VIEW_DIR, DATASET_DEF,
    train_split="train", val_split="val",
    dest=WORK / "data.yaml",
)
print("data.yaml:\n" + DATA_YAML.read_text())

### Sanity check — draw one sample with its boxes

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

colors = {0: "lime", 1: "red"}

img_paths = sorted((DATASET_DIR / "images" / "train").glob("*.png"))
sample = random.choice(img_paths)
img = Image.open(sample)
W, H = img.size
label = (DATASET_DIR / "labels" / "train" / f"{sample.stem}.txt").read_text().splitlines()

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img)
for line in label:
    cls, cx, cy, w, h = line.split()
    cls = int(cls); cx, cy, w, h = map(float, (cx, cy, w, h))
    x0 = (cx - w / 2) * W; y0 = (cy - h / 2) * H
    ax.add_patch(patches.Rectangle((x0, y0), w * W, h * H,
                                   fill=False, edgecolor=colors.get(cls, "yellow"), lw=1.5))
ax.set_title(f"{sample.name}  ({len(label)} boxes)")
ax.axis("off")
plt.show()

## 4 · Train

YOLOv7 trains from its COCO-pretrained checkpoint. `nc` is taken from
`data.yaml`, so the stock `cfg` files need no editing. Kaggle's ~12 h budget
caps `EPOCHS`; lower `BATCH` if you hit OOM at `img=1024`.

`WANDB_MODE=disabled` stops YOLOv7's Weights & Biases logger from prompting for
a login (wandb is preinstalled on Kaggle); training still writes its own
`results.txt` / curves under `runs/`.

In [ ]:
WEIGHTS_URL = f"https://github.com/WongKinYiu/yolov7/releases/download/v0.1/{WEIGHTS}"
!wget -nc {WEIGHTS_URL} -P yolov7/
!ls -lh yolov7/{WEIGHTS}

In [ ]:
!cd yolov7 && WANDB_MODE=disabled python train.py \
  --workers 4 --device 0 \
  --batch-size {BATCH} \
  --data {DATA_YAML} \
  --img {IMG} {IMG} \
  --cfg {CFG} \
  --weights {WEIGHTS} \
  --hyp {HYP} \
  --epochs {EPOCHS} \
  --name {RUN_NAME} \
  --project {RUNS_DIR} \
  --exist-ok

## 5 · Evaluate on `val` and compare to Table 5

`test.py` reports per-class AP at the COCO IoU sweep (mAP) plus mAP₅₀ / mAP₇₅.

In [ ]:
BEST = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
!cd yolov7 && WANDB_MODE=disabled python test.py \
  --data {DATA_YAML} \
  --img {IMG} \
  --batch {BATCH} \
  --conf 0.001 --iou 0.65 \
  --task val \
  --weights {BEST} \
  --name {RUN_NAME}_eval \
  --project {RUNS_DIR} \
  --exist-ok \
  --verbose

Paper reference (hidden test set) for side-by-side reading:

```
YOLOv7   mAP 60.48   mAP50 82.47   mAP75 62.30   AP crop 83.06   AP weed 37.91
```

Our `val` numbers print above. Differences are expected from: val vs hidden
test, the do-not-care partial-instance rule, and (for `tiny`) the smaller model
+ tiling.

## 6 · Package artifact → `/kaggle/working`

Keeps the trained weights, `data.yaml`, training curves and a small manifest
(the image tree lives in the mounted dataset, so nothing bulky is republished).

In [ ]:
import json
import shutil

ART = WORK / "artifact"
ART.mkdir(parents=True, exist_ok=True)

run_dir = RUNS_DIR / RUN_NAME
for fname in ["weights/best.pt", "results.txt", "results.png",
              "confusion_matrix.png", "PR_curve.png"]:
    src = run_dir / fname
    if src.exists():
        dst = ART / Path(fname).name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)

shutil.copy(DATA_YAML, ART / "data.yaml")

manifest = {
    "task": "object_detection",
    "model": "yolov7" if VARIANT == "full" else "yolov7-tiny",
    "variant": VARIANT,
    "dataset": "phenobench_multiclass",
    "classes": class_names,
    "img_size": IMG,
    "dataset_dir": str(DATASET_DIR),
    "epochs": EPOCHS,
    "batch": BATCH,
    "paper_baseline_testset": {
        "mAP": 60.48, "mAP50": 82.47, "mAP75": 62.30,
        "AP_crop": 83.06, "AP_weed": 37.91,
    },
}

# Carry over how the mounted dataset was built, if the export wrote a manifest.
export_manifest = YOLO_DS_ROOT / "export_manifest.json"
if export_manifest.exists():
    manifest["dataset_export"] = json.loads(export_manifest.read_text())

(ART / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Artifact contents:")
for p in sorted(ART.rglob("*")):
    print(" ", p.relative_to(ART))